# Mathematical Correctness Proof `core_logistic_regression_lasso.py`

The cost function is:
$$J(w) = L(w) + \alpha \|w\|_1$$
where $L(w)$ is the weighted logistic loss and $\alpha \|w\|_1$ is the L1-regularization.

We prove that `fit` correctly executes **Proximal Gradient Descent** on $J(w)$. This means: the model should fit well ($L$ small) but not become too complex ($|w|_1$ small).



----------

## Loop Invariant (I)

At the beginning of the $t$-th iteration the following holds: After iteration $t$, the code must have the same number in self.coef_ that is also mathematically computed as $w^{(t)}$.

> `self.coef_` $= w^{(t)}$ is the result of exactly $t$ correct Proximal Gradient steps, for all $k=1, \ldots, t$:
> $$z^{(k)} = w^{(k-1)} - \eta \, \nabla L(w^{(k-1)})$$
> $$w^{(k)} = S_{\eta\alpha}(z^{(k)})$$

----------

## IA - Initialization ($t = 0$)

**To show:** The invariant holds before the first iteration.

The code then sets:
```python
self.coef_      = np.zeros(n_features)   # w^(0) = 0
self.intercept_ = 0.0                    # b^(0) = 0
```

For $t = 0$ the invariant is empty, no steps have been executed yet. The statement is therefore true. 

---------

## IS - Maintenance ($t \to t+1$)

**IH:** `self.coef_` $= w^{(t)}$ is the result of $t$ correct steps. We therefore assume that the invariant holds for $t$.

**To show:** After executing the loop body, `self.coef_` contains the value $w^{(t+1)}$ according to the invariant. We must therefore show that the invariant also holds after $t+1$.

The update consists of two sub-steps:

**IS)a) Gradient**:
```python
p_hat    = self._sigmoid(X @ self.coef_ + self.intercept_)
residual = p_hat - y
grad_coef = (X.T @ (sw * residual)) / sw_sum
coef_new  = self.coef_ - self.learning_rate * grad_coef
```
The gradient of the logistic loss with respect to $w$ is:
$$\nabla L(w^{(t)}) = \frac{X^\top (\hat{p}^{(t)} - y)}{n} \qquad \text{(weighted: } \frac{X^\top(w \odot (\hat{p}^{(t)} - y))}{\sum_i w_i}\text{)}$$
The code computes exactly this gradient and sets:
$$\texttt{coef\_new} = w^{(t)} - \eta \, \nabla L(w^{(t)}) = z^{(t+1)} $$
(TRUE)

**IS)b) Proximal mapping**:
```python
coef_new = self._soft_threshold(coef_new, self.alpha * self.learning_rate)
```
The soft-thresholding operator $S_\lambda(z) = \operatorname{sign}(z) \cdot \max(|z| - \lambda, 0)$ is the analytical solution of the prox-problem:
$$S_\lambda(z) = \arg\min_w \frac{1}{2}\|w - z\|^2 + \lambda\|w\|_1$$
With $\lambda = \eta\alpha$ the code sets:
$$\texttt{coef\_new} = S_{\eta\alpha}(z^{(t+1)}) = w^{(t+1)} $$

(TRUE)
Since $L(w)$ is convex and Lipschitz-continuously differentiable, for sufficiently small `learning_rate`:
$$J(w^{(t+1)}) \leq J(w^{(t)})$$
i.e. each iteration approaches the global minimum.

Subsequently `self.coef_` $= w^{(t+1)}$ is set, the invariant holds for $t+1$. (TRUE)

--------

## Termination

The loop terminates in two cases:

**Case 1:CConvergence** :
```python
if delta < self.tol:
    break
```
with $\delta = \|w^{(t+1)} - w^{(t)}\|_\infty$. At this point the change is minimal and we are at a minimum that corresponds to the global minimum of $J(w)$. (TRUE)

**Case 2: Maximum Iterations** :
```python
for iteration in range(self.max_iter):
```
The loop terminates after `max_iter` steps. In this case the algorithm returns $w^{(T)}$ and warns via `ConvergenceWarning`. According to the invariant, $w^{(T)}$ is the result of $T$ correct steps, however the convergence described in Case 1 is not fulfilled. Possible code abort is also covered by this case. 

--------

## Conclusion

Since the invariant holds at the beginning (IA) and is maintained after each step (IS), fit correctly computes Proximal Gradient Descent on $J(w)$. The loop terminates in every case, and the result self.coef_ corresponds to the global minimum at convexity.
Fertig